First we gotta explore the dataset we dropped in our volume, so we can understand the datasets structures, volume and possible data quality issues before we head into the silver layer.

In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/olist_lakehouse/landing/olist/historical"

In [0]:
df_orders = (
    spark.read
    .option("header", True)       # analyze the df head
    .option("inferSchema", True)  # Spark tries to infer the column types
    .csv(f"{BASE_PATH}/olist_orders_dataset.csv")
)

display(df_orders)

In [0]:
df_orders.printSchema()

In [0]:
# getting col and row counts:

print(f"Number of columns: {len(df_orders.columns)}")
print(f"Number of rows: {df_orders.count()}")

In [0]:
# exploring basic transformation spark functions

df_cancelled = (
    df_orders
    .select("order_id", "customer_id", "order_status", "order_purchase_timestamp")
    .filter(F.col("order_status") == "canceled")

)

df_cancelled.show()

In [0]:
# grouping and counting

df_grouped = (
    df_orders
    .groupBy("order_status")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

df_grouped.show()

In [0]:
df_delivery = df_orders.withColumn(
    "days until delivery",
    F.datediff(
        F.col("order_delivered_customer_date"),
        F.col("order_purchase_timestamp")
    )
)

df_delivery.show()   

In [0]:
df_delivery.select("order_id","order_status").show()

In [0]:
# data quality analysis

display(
    df_orders.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_orders.columns]
    )
)

In [0]:
# range of the dataset

display(
    df_orders.agg(
        F.min(df_orders.order_purchase_timestamp).alias('Start date'),
        F.max(df_orders.order_purchase_timestamp).alias('End date')
    )
)

In [0]:
# reading customers csv

df_customers = (
    spark.read
    .option("header", True)       # analyze the df head
    .option("inferSchema", True)  # Spark tries to infer the column types
    .csv(f"{BASE_PATH}/olist_customers_dataset.csv")
)

display(df_customers)

In [0]:
# joining orders and customers datasets

display(
    df_orders
    .join(
        df_customers,
        df_orders.customer_id == df_customers.customer_id
    )
    .groupBy("customer_state")
    .agg(F.count("order_id").alias("orders_qty"))
    .orderBy(F.col("orders_qty").desc()
)
)

In [0]:
# reviews dataset

path_reviews = f"{BASE_PATH}/olist_order_reviews_dataset.csv"

df_reviews = (
    spark.read
    .option("header", True)       # analyze the df head
    .option("multiLine", True)  # some strings on comments column has multiple lines and spark creates new records for them
    .option("escape", '"') #some records has double quotes and spark creates new records for them
    .option("inferSchema", True)  # Spark tries to infer
    .csv(path_reviews)

)

display(df_reviews)

In [0]:
# exploring it on sql
df_orders.createOrReplaceTempView("orders")

In [0]:
%sql
select * from orders